## Imports

In [2]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [3]:
import numpy as np

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [5]:
from ucimlrepo import fetch_ucirepo

In [6]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [7]:
X = X.fillna(value=0)

In [8]:
y

,num
0,0
1,2
2,1
3,0
4,0
...,...
298,1
299,2
300,3
301,1


In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [10]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [11]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [12]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [13]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [14]:
y_train.dtype

torch.int64

In [15]:
x_train = x_train.type(torch.float64)
x_test = x_test.type(torch.float64)

In [16]:
loader = data.DataLoader(
    data.TensorDataset(
        x_train, 
        y_train), 
    batch_size = 4, 
    shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [17]:
y_trainset.dtype

torch.int64

## Model & Training

In [18]:
2**13

8192

In [19]:
model = nft.h_ANFIS(
    input_size = 13,
    num_mfs = 7,
    outputs = 5,
    #membership_function=nft.Gaussian_MF,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float64
)

#model.init_premises(x_trainset)

model.init_consequents(x_trainset, y_trainset)

In [20]:
epochs=1000

loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

validation=0.3

early_stopping = nft.EarlyStopping(
    patience=30, 
    delta=0.0001
)

In [21]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=validation,
    early_stopping=early_stopping
)

In [22]:
trainer(model, loader, verbose=True)

Epoch:    1/1000 - loss: 2.100398 - validation loss: 2.057425
Epoch:    2/1000 - loss: 2.047527 - validation loss: 2.023587
Epoch:    3/1000 - loss: 2.003961 - validation loss: 1.993885
Epoch:    4/1000 - loss: 1.980608 - validation loss: 1.975297
Epoch:    5/1000 - loss: 1.950006 - validation loss: 1.951066
Epoch:    6/1000 - loss: 1.934175 - validation loss: 1.938331
Epoch:    7/1000 - loss: 1.912753 - validation loss: 1.918594
Epoch:    8/1000 - loss: 1.893580 - validation loss: 1.904786
Epoch:    9/1000 - loss: 1.880557 - validation loss: 1.895255
Epoch:   10/1000 - loss: 1.856474 - validation loss: 1.873707
Epoch:   11/1000 - loss: 1.840279 - validation loss: 1.860401
Epoch:   12/1000 - loss: 1.824337 - validation loss: 1.847098
Epoch:   13/1000 - loss: 1.802903 - validation loss: 1.822105
Epoch:   14/1000 - loss: 1.788431 - validation loss: 1.813960
Epoch:   15/1000 - loss: 1.776798 - validation loss: 1.807704
Epoch:   16/1000 - loss: 1.760397 - validation loss: 1.790640
Epoch:  

In [23]:
train_measures = nft.get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.6698113207547169
Precision: 0.6588928724900833
Recall: 0.6698113207547169
F1: 0.6521955605632369
Confusion Matrix: [[104   6   1   3   1]
 [ 18  15   2   3   0]
 [  8   5  10   2   0]
 [  5   8   0  11   1]
 [  0   3   0   4   2]]


In [24]:
test_measures = nft.get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.46153846153846156
Precision: 0.47983147287495104
Recall: 0.46153846153846156
F1: 0.46705775427579943
Confusion Matrix: [[34  8  5  2  0]
 [ 9  4  1  2  1]
 [ 1  7  0  3  0]
 [ 2  4  1  3  0]
 [ 0  2  0  1  1]]
